# S0.1P · 离线比较的正例下界、小人群可检出效应与市场正例上限

**问题**：S0.1 给出了随机对照增益检验所需的人数（N_eff ≥ 30,000）。恢复实验前还缺三个量：

1. 离线比较两个模型的 ROC曲线下面积（AUROC）时，最少需要多少**正例**；
2. 可用人群只有几千人时，双臂随机对照**最小能检出多大的效应**；
3. 用公开统计给每个观察窗内的**正例数设上限**。

方法：Hanley–McNeil 近似 + 配对相关修正；双臂检验沿用 S0.1 的 `required_n_per_arm`。
参数全部来自 `modules/m0_compliance/configs/s0_1p_validation_power.yaml`，
逻辑在 `modules/m0_compliance/components/validation_power.py`。

> ⚠️ 近似公式用于定量级与排优先级，不替代实验后基于真实分数的配对 bootstrap 区间。

In [1]:
import hashlib
import pathlib
import subprocess
import sys

import pandas as pd
import yaml

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m0_compliance/configs/s0_1p_validation_power.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config_hash = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()[:HASH_PREFIX_LEN]
git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True,
                                  cwd=REPO_ROOT).strip()
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + config_hash)
print("seed:", cfg["seed"])
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", cfg["step_id"], "|", cfg["purpose"])

config: modules/m0_compliance/configs/s0_1p_validation_power.yaml sha256:487fe0aae38c
seed: 42
git: e320d91（工作区含本步未提交改动）
step: S0.1P | 补算 S0.1 未覆盖的三个量：离线 AUROC 比较的正例下界、几千人规模下的可检出增益、观察窗内正例数上限


In [2]:
from modules.m0_compliance.components.validation_power import positives_table

PERCENT = 100
RESULTS = REPO_ROOT / "modules/m0_compliance/results"

rows = []
for auc in cfg["reference_aucs"]:
    rows += positives_table(auc, cfg["paired_auc_mdes"], cfg["base_rates"],
                            cfg["estimate_correlations"], cfg["alpha"], cfg["power"])
positives = pd.DataFrame(rows)
positives.to_csv(RESULTS / "offline_auc_positives.csv", index=False)

view = positives.copy()
view["正例率"] = (view["base_rate"] * PERCENT).round(1).astype(str) + "%"
view = view.rename(columns={"auc": "参考AUROC", "paired_auc_mde": "待检出配对差",
                            "estimate_correlation": "估计相关",
                            "min_positives": "最少正例数", "implied_customers": "折合客户数"})
print(view[["参考AUROC", "待检出配对差", "正例率", "估计相关", "最少正例数", "折合客户数"]]
      .to_string(index=False))

 参考AUROC  待检出配对差  正例率  估计相关  最少正例数   折合客户数
    0.86   0.020 3.0%   0.8    441   14700
    0.86   0.010 3.0%   0.8   1763   58767
    0.86   0.005 3.0%   0.8   7049  234967
    0.86   0.020 3.0%   0.9    221    7367
    0.86   0.010 3.0%   0.9    882   29400
    0.86   0.005 3.0%   0.9   3525  117500
    0.86   0.020 1.0%   0.8    439   43900
    0.86   0.010 1.0%   0.8   1753  175300
    0.86   0.005 1.0%   0.8   7010  701000
    0.86   0.020 1.0%   0.9    220   22000
    0.86   0.010 1.0%   0.9    877   87700
    0.86   0.005 1.0%   0.9   3505  350500
    0.70   0.020 3.0%   0.8    691   23034
    0.70   0.010 3.0%   0.8   2762   92067
    0.70   0.005 3.0%   0.8  11048  368267
    0.70   0.020 3.0%   0.9    346   11534
    0.70   0.010 3.0%   0.9   1381   46034
    0.70   0.005 3.0%   0.9   5524  184134
    0.70   0.020 1.0%   0.8    683   68300
    0.70   0.010 1.0%   0.8   2731  273100
    0.70   0.005 1.0%   0.8  10921 1092100
    0.70   0.020 1.0%   0.9    342   34200
    0.70   

**读法**：最少正例数几乎不随正例率变化（同一行在 3.0% 与 1.0% 下相差不到 2%），
变化的只是折合客户数。所以离线比较两个模型时，判断能否检出差异要看**正例数**。

- 在估计相关 0.8 的保守档、参考 AUROC 0.86 下：检出配对差 0.02 约需 440 个正例，0.01 约需 1,760 个，0.005 约需 7,050 个。
- 参考 AUROC 0.70 时需要更多正例（单个 AUROC 的方差更大）：检出 0.02 约需 690 个。
- 这里的正例数都指**测试集内**的正例，不是全部样本的正例。
- 估计相关从 0.8 升到 0.9，所需正例约减半。这个相关系数是假设，实验后应以真实分数的配对 bootstrap 区间为准。

In [3]:
from modules.m0_compliance.components.validation_power import small_population_table

small = pd.DataFrame(small_population_table(cfg["primary_base_rate"], cfg["populations"],
                                            cfg["alpha"], cfg["power"], cfg["holdout_share"]))
small.to_csv(RESULTS / "small_population_detectable_lift.csv", index=False)

view = small.copy()
view["可检出相对增益"] = view["min_relative_lift"].map(
    lambda v: "检不出" if pd.isna(v) else "%.1f%%" % (v * PERCENT))
view["可检出绝对增益"] = view["min_absolute_lift"].map(
    lambda v: "检不出" if pd.isna(v) else "+%.2f 个百分点" % (v * PERCENT))
view = view.rename(columns={"scenario": "人群档位", "n_total": "总人群",
                            "expected_control_events": "对照臂期望转化数",
                            "normal_approx_valid": "正态近似可用"})
print(view[["人群档位", "总人群", "对照臂期望转化数", "正态近似可用", "可检出相对增益", "可检出绝对增益"]]
      .to_string(index=False))

      人群档位   总人群  对照臂期望转化数  正态近似可用 可检出相对增益     可检出绝对增益
   R-A_low     3      0.03   False 2972.2% +89.17 个百分点
   R-A_mid   226      3.39   False  330.6%  +9.92 个百分点
  R-A_high  5814     87.21    True   46.2%  +1.38 个百分点
S0.1_N_eff 30000    450.00    True   19.2%  +0.58 个百分点


**读法**：与 S0.1 同一检验口径（基线 3.0%、双侧 α=0.05、功效 0.80、双臂等分）。

- S0.1 的 30,000 人档可检出约 19% 的相对增益，与 S0.1「相对增益 20%」的口径一致（交叉核对通过）。
- S0.5 漏斗最乐观档 5,814 人只能检出约 46% 的相对增益（绝对 +1.4 个百分点）。
- 226 人和 3 人两档的对照臂期望转化数不到 5 个，两比例检验的正态近似本身不成立，表中数字只说明「检不出」，不能当作可检出效应引用。

In [4]:
from modules.m0_compliance.components.validation_power import market_positive_ceiling

stat = cfg["market_statistic"]
print("公开统计：", stat["name"], "=", stat["value"], "｜来源：", stat["source"])
print("口径提醒：", stat["caveat"])
ceiling = pd.DataFrame(market_positive_ceiling(stat["value"], stat["months_observed"],
                                               cfg["window_days"], cfg["institution_shares"]))
ceiling.to_csv(RESULTS / "market_positive_ceiling.csv", index=False)

view = ceiling.copy()
view["机构份额（情景）"] = (view["institution_share"] * PERCENT).round(0).astype(int).astype(str) + "%"
view["全市场年化新增"] = view["market_new_per_year"].round(0).astype(int)
view["每个观察窗全市场新增"] = view["market_new_per_window"].round(0).astype(int)
view["每个观察窗该机构正例上限"] = view["institution_positive_ceiling_per_window"].round(0).astype(int)
print(view[["机构份额（情景）", "全市场年化新增", "每个观察窗全市场新增", "每个观察窗该机构正例上限"]]
      .to_string(index=False))

公开统计： 2026 年 1–8 月跨境理财通内地新增个人投资者 = 8800 ｜来源： 中国金融信息网 2026-09-30 报道（文中称来自央行数据）
口径提醒： 「新增投资者」是首次参与该业务，不等于「首次成功申购」；是全市场全部银行的合计
机构份额（情景）  全市场年化新增  每个观察窗全市场新增  每个观察窗该机构正例上限
      5%    13200        3255           163
     10%    13200        3255           325
     20%    13200        3255           651


## 结论

1. **离线比较看正例数。** S0.1 的 N_eff ≥ 30,000 是随机对照增益检验的人数口径，本身没有错；
   但它不能直接回答「两个模型的 AUROC 差能不能被检出」。后者由正例数决定：
   检出 0.02 的配对差约需 440 个正例，检出 0.01 约需 1,760 个（参考 AUROC 0.86、估计相关 0.8）。
2. **几千人规模的试点只能检出大效应。** 5,814 人时可检出相对增益约 46%，
   远大于 DR-MX-001 · D-4 的 20%。对 226 人和 3 人两档，检验的正态近似本身不成立。
3. **正例数有公开上限。** 按 2026 年 1–8 月全市场内地新增投资者约 8,800 人折算，
   每个 90 日观察窗全市场约 3,250 人；即使假设某机构占 20%，单窗正例上限约 650 个。
   上表的正例数指**测试集内**的正例：若测试集只占全部样本的 20%–30%，单窗能进测试集的正例只有约 130–200 个，
   连参考 AUROC 0.86 下 0.02 的配对差（约 440 个）都检不出，需要累积多个观察窗。
   新增投资者 ≠ 首次成功申购，所以这是**上限**，真实正例只会更少。

> ⚠️ 份额、AUROC、估计相关均为情景假设；基线转化率 3.0% 仍是 DR-MX-001 · D-3 的显式假设。
> 本步只做公式推导，不使用任何真实客户数据，结论不得外推为汇丰业务表现。